# Gaia XP Features and some engineering: coefficient errors, SNR features, and XP summary flags

This notebook augments an existing classification dataset with additional Gaia XP-derived feature blocks:

- Coefficient uncertainties: `bp_coefficient_errors` and `rp_coefficient_errors` (per source, per coefficient).
- Signal-to-Noise-ratio-style features: compact signal-to-noise descriptors derived from coefficients and their uncertainties.
- XP summary quality indicators: optional quality/availability flags that can be used for filtering or as features.

The output is a CSV of the input classification table, augmented with these new columns.

## Requirements

- An input classification CSV (`binary_vosa_XPcoeff_110d_L2_derivatives.csv`)


# Add Gaia XP uncertainty (errors), SNR-style features and xp_summary indicators

This notebook duplicates an existing CSV that already contains:
- `c000..c109` (original 110D vector)
- `d000..d109` (derivative 110D vector)
- `y` label

and appends:

**(a)** `bp_coefficient_errors`, `rp_coefficient_errors` (expanded to 110 columns)
**(b)** SNR-type features (`|coeff| / error`) (expanded to 110 columns + summary stats)
**(c)** `gaiadr3.xp_summary` auxiliary quality indicators.

It uses the same Gaia@AIP auth + Simple Join Service (SJS) pattern as `gaia_aip_pavyzdinis.ipynb`.

In [1]:
# pip install requests pandas astropy pyvo python-dotenv

In [2]:
import os
import re
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import pyvo as vo
from astropy.io.votable import parse_single_table


In [ ]:
# 1) Locate the input CSV
print("CWD:", Path.cwd())

ROOT = Path.cwd().resolve()
OUT_DIR = ROOT / "out_data"
if not OUT_DIR.exists():
    hits = list(ROOT.rglob("out_data"))
    if hits:
        OUT_DIR = hits[0]
print("OUT_DIR:", OUT_DIR)

# Optional manual override:
INPUT_CSV = None  # e.g. OUT_DIR / "binary_vosa_XPcoeff_110d_L2_derivatives.csv"

def _looks_like_merged_ml_csv(path: Path) -> bool:
    try:
        df0 = pd.read_csv(path, nrows=1)
    except Exception:
        return False
    cols = set(df0.columns)
    has_ids = "source_id" in cols and "y" in cols
    has_c = ("c000" in cols) and ("c109" in cols)
    has_d = ("d000" in cols) and ("d109" in cols)
    return has_ids and has_c and has_d

if INPUT_CSV is None:
    candidates = sorted([p for p in OUT_DIR.glob("*.csv") if _looks_like_merged_ml_csv(p)],
                        key=lambda p: p.stat().st_mtime,
                        reverse=True)
    if not candidates:
        raise FileNotFoundError(
            f"Neradau tinkamo CSV in {OUT_DIR}. Reikia, kad būtų: source_id, y, c000..c109, d000..d109.\n"
            "Nurodyk ranka INPUT_CSV."
        )
    if len(candidates) > 1:
        print("Found multiple candidates. Picking the newest. Candidates:")
        for p in candidates[:10]:
            print(" -", p.name)
    INPUT_CSV = candidates[0]

print("INPUT_CSV:", INPUT_CSV)

df_base = pd.read_csv(INPUT_CSV)
df_base["source_id"] = pd.to_numeric(df_base["source_id"], errors="coerce").astype("Int64")
df_base = df_base.dropna(subset=["source_id"]).copy()
df_base["source_id"] = df_base["source_id"].astype("int64")

source_ids = np.sort(df_base["source_id"].unique())
print("Rows:", len(df_base), "| unique source_id:", len(source_ids))
df_base.head()


In [ ]:
# 2) Auth
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

GAIA_AIP_TOKEN = os.getenv("GAIA_AIP_TOKEN", "YOUR_TOKEN_GOES_HERE").strip()
#                                             ^^^^^^^^^^^^^^^^^^^^
if not GAIA_AIP_TOKEN:
    token_file = Path(".gaia_aip_token")
    if token_file.exists():
        GAIA_AIP_TOKEN = token_file.read_text(encoding="utf-8").strip()

if GAIA_AIP_TOKEN and GAIA_AIP_TOKEN[0] in ("'", '"') and GAIA_AIP_TOKEN[-1] == GAIA_AIP_TOKEN[0]:
    GAIA_AIP_TOKEN = GAIA_AIP_TOKEN[1:-1].strip()

if not GAIA_AIP_TOKEN:
    raise RuntimeError(
        "Set GAIA_AIP_TOKEN (env var) or create .gaia_aip_token file containing: Token <your_token>\n"
        "Then restart the kernel."
    )

GAIA_AIP_TOKEN = GAIA_AIP_TOKEN if GAIA_AIP_TOKEN.startswith("Token ") else f"Token {GAIA_AIP_TOKEN}"

TAP_URL = "https://gaia.aip.de/tap"
SJS_URL = "https://gaia.aip.de/uws/simple-join-service"

sess = requests.Session()
sess.headers["Authorization"] = GAIA_AIP_TOKEN

print("Token prefix OK:", GAIA_AIP_TOKEN.split()[0])


In [5]:
# 3) Helpers (TAP async + SJS)
def read_first_votable(path: Path) -> pd.DataFrame:
    tab = parse_single_table(str(path)).to_table(use_names_over_ids=True)
    return tab.to_pandas()

def tap_create_async_job(session: requests.Session, query: str, *, lang: str = "ADQL", runid: str = "ids_for_sjs") -> str:
    url = f"{TAP_URL}/async"
    query = query.strip().rstrip(";")
    payload = {
        "REQUEST": "doQuery",
        "LANG": lang,
        "FORMAT": "votable",
        "QUERY": query,
        "RUNID": runid,
    }
    r = session.post(url, data=payload, allow_redirects=False, timeout=120)

    if r.status_code == 403 and "Invalid token" in (r.text or ""):
        raise RuntimeError("HTTP 403: Invalid token. Check GAIA_AIP_TOKEN and restart kernel.")

    if r.status_code in (302, 303) and "Location" in r.headers:
        job_url = r.headers["Location"]
        if job_url.startswith("/"):
            job_url = "https://gaia.aip.de" + job_url
    else:
        raise RuntimeError(f"Unexpected TAP async response: HTTP {r.status_code}; body={r.text[:300]!r}")

    job_id = job_url.rstrip("/").split("/")[-1]
    session.post(f"{job_url}/phase", data={"PHASE": "RUN"}, timeout=60).raise_for_status()

    t0 = time.time()
    while True:
        ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
        if ph in ("COMPLETED", "ERROR", "ABORTED"):
            break
        if time.time() - t0 > 180:
            raise TimeoutError("TAP async job timed out (>180s).")
        time.sleep(1.5)

    if ph != "COMPLETED":
        raise RuntimeError(f"TAP async job ended with phase={ph}")

    return job_id

def tap_async_query_to_df(session: requests.Session, query: str, *, runid: str = "tap_query") -> pd.DataFrame:
    # create
    url = f"{TAP_URL}/async"
    q = query.strip().rstrip(";")
    payload = {"REQUEST":"doQuery","LANG":"ADQL","FORMAT":"votable","QUERY":q,"RUNID":runid}
    r = session.post(url, data=payload, allow_redirects=False, timeout=120)

    if r.status_code == 403 and "Invalid token" in (r.text or ""):
        raise RuntimeError("HTTP 403: Invalid token. Check GAIA_AIP_TOKEN and restart kernel.")

    if r.status_code in (302, 303) and "Location" in r.headers:
        job_url = r.headers["Location"]
        if job_url.startswith("/"):
            job_url = "https://gaia.aip.de" + job_url
    else:
        raise RuntimeError(f"Unexpected TAP async response: HTTP {r.status_code}; body={r.text[:300]!r}")

    # run
    session.post(f"{job_url}/phase", data={"PHASE":"RUN"}, timeout=60).raise_for_status()

    t0 = time.time()
    while True:
        ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
        if ph in ("COMPLETED","ERROR","ABORTED"):
            break
        if time.time() - t0 > 240:
            raise TimeoutError("TAP async job timed out (>240s).")
        time.sleep(1.5)

    if ph != "COMPLETED":
        raise RuntimeError(f"TAP async job ended with phase={ph}")

    # download
    res_url = f"{job_url}/results/result"
    content = session.get(res_url, timeout=300).content
    tab = parse_single_table(io.BytesIO(content)).to_table(use_names_over_ids=True)
    return tab.to_pandas()

def sjs_download(session: requests.Session, job_id: str, join_table: str, *,
                 column_name: str = "source_id",
                 responseformat: str = "votable",
                 data_structure: str = "COMBINED") -> Path:
    service = vo.dal.DALService(SJS_URL, session=session)
    q = service.create_query(
        job_id=job_id,
        column_name=column_name,
        responseformat=responseformat,
        join_table=join_table,
        data_structure=data_structure,
    )
    resp = q.submit(post=True)
    resp.raise_for_status()
    job = vo.io.uws.parse_job(io.BytesIO(resp.content))

    service._session.post(f"{service._baseurl}/{job.jobid}/phase", data={"PHASE": "RUN"}, stream=True).raise_for_status()

    t0 = time.time()
    while True:
        ph = service._session.get(f"{service._baseurl}/{job.jobid}/phase").text.strip()
        if ph in ("COMPLETED", "ERROR", "ABORTED"):
            break
        if time.time() - t0 > 240:
            raise TimeoutError("SJS job timed out (>240s).")
        time.sleep(1.5)

    if ph != "COMPLETED":
        raise RuntimeError(f"SJS ended with phase={ph}")

    job_url = f"{service._baseurl}/{job.jobid}"
    job2 = vo.io.uws.parse_job(io.BytesIO(service._session.get(job_url).content))

    results = job2.results
    if hasattr(results, "keys") and callable(getattr(results, "keys")):
        first_key = sorted(list(results.keys()))[0]
        href = results[first_key].href
        key = str(first_key)
    else:
        res_list = list(results)
        if not res_list:
            raise RuntimeError("SJS job has no results.")
        href = res_list[0].href
        key = "result"

    out_path = OUT_DIR / f"tmp_sjs_{job2.jobid}_{key}.vot"

    last = None
    for attempt in range(1, 5):
        r = service._session.get(href, timeout=300)
        r.raise_for_status()
        content = r.content
        last = content

        # basic sanity checks for a VOTable payload
        ok = (content is not None) and (len(content) > 500) and (b"VOTABLE" in content[:5000]) and (b"</VOTABLE>" in content[-2000:])
        if ok:
            out_path.write_bytes(content)
            return out_path

        time.sleep(1.0 * attempt)

    out_path.write_bytes(last or b"")
    raise RuntimeError(
        "Downloaded SJS result looks truncated / not a valid VOTable. "
        f"Saved payload to: {out_path}. Try re-running the cell."
    )


In [ ]:
# 4) Fetch XP continuous arrays (coefficients + coefficient_errors) via SJS
ID_CHUNK_SIZE = 300

def fetch_xp_continuous_for_ids(ids: np.ndarray) -> pd.DataFrame:
    ids_sql = ",".join(str(int(sid)) for sid in ids)
    q_ids_job = f"""
    SELECT source_id
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_sql})
    """
    tap_job_id = tap_create_async_job(sess, q_ids_job, runid="ids_for_sjs")
    path_vot = sjs_download(
        sess,
        tap_job_id,
        "gaiadr3.xp_continuous_mean_spectrum",
        responseformat="votable",
        data_structure="COMBINED",
    )
    df = read_first_votable(path_vot)
    keep = ["source_id",
            "bp_coefficients", "bp_coefficient_errors",
            "rp_coefficients", "rp_coefficient_errors"]
    keep = [c for c in keep if c in df.columns]
    return df[keep].copy()

dfs = []
for i0 in range(0, len(source_ids), ID_CHUNK_SIZE):
    chunk = source_ids[i0:i0+ID_CHUNK_SIZE]
    print(f"SJS chunk {i0//ID_CHUNK_SIZE+1}: ids={len(chunk)}")
    df_xp = fetch_xp_continuous_for_ids(chunk)
    print("  rows:", len(df_xp))
    if len(df_xp):
        dfs.append(df_xp)

df_xp_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["source_id"])
df_xp_all["source_id"] = pd.to_numeric(df_xp_all["source_id"], errors="coerce").astype("Int64")
df_xp_all = df_xp_all.dropna(subset=["source_id"]).copy()
df_xp_all["source_id"] = df_xp_all["source_id"].astype("int64")
df_xp_all = df_xp_all.drop_duplicates("source_id")

print("XP continuous rows fetched:", len(df_xp_all), "out of", len(source_ids))
df_xp_all.head()


In [ ]:
# 5) Fetch xp_summary quality indicators via TAP (scalars)
XP_SUMMARY_COLS = [
    "bp_n_relevant_bases", "bp_relative_shrinking", "bp_n_measurements", "bp_n_rejected_measurements",
    "bp_standard_deviation", "bp_chi_squared", "bp_n_transits", "bp_n_contaminated_transits", "bp_n_blended_transits",
    "rp_n_relevant_bases", "rp_relative_shrinking", "rp_n_measurements", "rp_n_rejected_measurements",
    "rp_standard_deviation", "rp_chi_squared", "rp_n_transits", "rp_n_contaminated_transits", "rp_n_blended_transits",
]

def fetch_xp_summary_for_ids(ids: np.ndarray) -> pd.DataFrame:
    ids_sql = ",".join(str(int(sid)) for sid in ids)
    cols_sql = ",\n  ".join(["source_id"] + XP_SUMMARY_COLS)
    q = f"""
    SELECT
      {cols_sql}
    FROM gaiadr3.xp_summary
    WHERE source_id IN ({ids_sql})
    """
    return tap_async_query_to_df(sess, q, runid="xp_summary")

dfs = []
for i0 in range(0, len(source_ids), ID_CHUNK_SIZE):
    chunk = source_ids[i0:i0+ID_CHUNK_SIZE]
    print(f"TAP xp_summary chunk {i0//ID_CHUNK_SIZE+1}: ids={len(chunk)}")
    df_s = fetch_xp_summary_for_ids(chunk)
    print("  rows:", len(df_s))
    if len(df_s):
        dfs.append(df_s)

df_sum = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=["source_id"])
df_sum["source_id"] = pd.to_numeric(df_sum["source_id"], errors="coerce").astype("Int64")
df_sum = df_sum.dropna(subset=["source_id"]).copy()
df_sum["source_id"] = df_sum["source_id"].astype("int64")
df_sum = df_sum.drop_duplicates("source_id")

print("xp_summary rows fetched:", len(df_sum), "out of", len(source_ids))
df_sum.head()


In [ ]:
# 6) Build features: errors (110) + SNR (110) + summary stats
def _to_float_array(x):
    # handle numpy arrays, masked arrays, lists
    if x is None:
        return None
    try:
        arr = np.asarray(x, dtype=float)
        return arr
    except Exception:
        try:
            return np.asarray(list(x), dtype=float)
        except Exception:
            return None

def expand_array_column(df: pd.DataFrame, col: str, prefix: str, expected_len: int) -> pd.DataFrame:
    arr2d = np.full((len(df), expected_len), np.nan, dtype=float)
    for i, x in enumerate(df[col].values):
        a = _to_float_array(x)
        if a is None or a.ndim != 1 or len(a) != expected_len:
            continue
        arr2d[i, :] = a
    cols = [f"{prefix}_{k:02d}" for k in range(expected_len)]
    return pd.DataFrame(arr2d, columns=cols)

# infer expected lengths (usually 55)
def infer_len(series):
    for x in series.values:
        a = _to_float_array(x)
        if a is not None and a.ndim == 1:
            return int(len(a))
    return None

n_bp = infer_len(df_xp_all.get("bp_coefficients", pd.Series([], dtype=object)))
n_rp = infer_len(df_xp_all.get("rp_coefficients", pd.Series([], dtype=object)))

if n_bp is None or n_rp is None:
    raise RuntimeError("Could not infer coefficient vector length from xp_continuous_mean_spectrum.")
print("BP coeff len:", n_bp, "| RP coeff len:", n_rp)

# Expand errors
df_bp_err = expand_array_column(df_xp_all, "bp_coefficient_errors", "bp_err", expected_len=n_bp)
df_rp_err = expand_array_column(df_xp_all, "rp_coefficient_errors", "rp_err", expected_len=n_rp)

# Expand coefficients (for SNR)
df_bp_c = expand_array_column(df_xp_all, "bp_coefficients", "bp_c", expected_len=n_bp)
df_rp_c = expand_array_column(df_xp_all, "rp_coefficients", "rp_c", expected_len=n_rp)

# Compute per-coefficient SNR
eps = 1e-12
bp_snr = np.abs(df_bp_c.to_numpy()) / (df_bp_err.to_numpy() + eps)
rp_snr = np.abs(df_rp_c.to_numpy()) / (df_rp_err.to_numpy() + eps)

df_bp_snr = pd.DataFrame(bp_snr, columns=[f"bp_snr_{k:02d}" for k in range(n_bp)])
df_rp_snr = pd.DataFrame(rp_snr, columns=[f"rp_snr_{k:02d}" for k in range(n_rp)])

# Summary stats
def summarize(arr2d: np.ndarray, prefix: str) -> pd.DataFrame:
    return pd.DataFrame({
        f"{prefix}_mean": np.nanmean(arr2d, axis=1),
        f"{prefix}_std":  np.nanstd(arr2d, axis=1),
        f"{prefix}_min":  np.nanmin(arr2d, axis=1),
        f"{prefix}_max":  np.nanmax(arr2d, axis=1),
        f"{prefix}_median": np.nanmedian(arr2d, axis=1),
    })

df_stats = pd.concat([
    summarize(df_bp_err.to_numpy(), "bp_err"),
    summarize(df_rp_err.to_numpy(), "rp_err"),
    summarize(bp_snr, "bp_snr"),
    summarize(rp_snr, "rp_snr"),
], axis=1)

# Assemble per-source feature table
df_feat = pd.concat([
    df_xp_all[["source_id"]].reset_index(drop=True),
    df_bp_err.reset_index(drop=True),
    df_rp_err.reset_index(drop=True),
    df_bp_snr.reset_index(drop=True),
    df_rp_snr.reset_index(drop=True),
    df_stats.reset_index(drop=True),
], axis=1)

print("Feature table rows:", len(df_feat), "| cols:", df_feat.shape[1])
df_feat.head()


In [ ]:
# Merge everything into a duplicate CSV
# Prefix xp_summary columns to avoid collisions
df_sum2 = df_sum.copy()
for c in XP_SUMMARY_COLS:
    if c in df_sum2.columns:
        df_sum2.rename(columns={c: f"xps_{c}"}, inplace=True)

df_merged = df_base.merge(df_feat, on="source_id", how="left")
df_merged = df_merged.merge(df_sum2[["source_id"] + [c for c in df_sum2.columns if c.startswith("xps_")]],
                            on="source_id", how="left")

# Coverage report
cov_xp = df_merged["bp_err_00"].notna().mean() if "bp_err_00" in df_merged.columns else 0.0
cov_sum = df_merged.get("xps_bp_n_transits", pd.Series([np.nan]*len(df_merged))).notna().mean()

print(f"Coverage: xp_continuous errors present for {cov_xp*100:.1f}% rows")
print(f"Coverage: xp_summary present for {cov_sum*100:.1f}% rows")

# Output path
out_name = INPUT_CSV.stem + "_errors_snr_xpsummary.csv"
OUT_CSV = INPUT_CSV.with_name(out_name)

df_merged.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)
df_merged.head()
